# Gold · Grain and relationship checks
Run after the Gold notebook and whenever the Silver inputs or selected period change. Raises an error if any Gold key or relationship fails.

In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "nyc_mobility", "Catalog")
dbutils.widgets.text("gold_schema", "nyc_gold", "Gold schema")
catalog = dbutils.widgets.get("catalog").strip()
gold_schema = dbutils.widgets.get("gold_schema").strip()
for identifier in (catalog, gold_schema):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Invalid catalog/schema: {identifier!r}")
fact = f"{catalog}.{gold_schema}.fact_taxi_trip"
zones = f"{catalog}.{gold_schema}.dim_zone"
weather = f"{catalog}.{gold_schema}.dim_weather_hour"

In [ ]:
checks = spark.sql(f"""
    SELECT COUNT(*) AS joined_rows,
           COUNT(DISTINCT f.trip_key) AS unique_trip_keys,
           COUNT_IF(p.location_id IS NULL) AS unmatched_pickup_zone,
           COUNT_IF(d.location_id IS NULL) AS unmatched_dropoff_zone,
           COUNT_IF(w.observation_hour IS NULL) AS unmatched_weather_hour
    FROM {fact} f
    LEFT JOIN {zones} p ON f.pickup_zone_id = p.location_id
    LEFT JOIN {zones} d ON f.dropoff_zone_id = d.location_id
    LEFT JOIN {weather} w ON f.weather_hour = w.observation_hour
""").first()
display(spark.createDataFrame([checks]))
fact_rows = spark.table(fact).count()
zone_rows = spark.table(zones).count()
weather_rows = spark.table(weather).count()
zone_ids = spark.table(zones).select("location_id").distinct().count()
weather_hours = spark.table(weather).select("observation_hour").distinct().count()
print(f"Fact: {fact_rows}; Zones: {zone_rows} ({zone_ids} keys); Weather: {weather_rows} ({weather_hours} keys)")
if (fact_rows == 0 or checks.joined_rows != fact_rows or
    checks.unique_trip_keys != fact_rows or zone_rows == 0 or zone_rows != zone_ids or
    weather_rows == 0 or weather_rows != weather_hours or any(checks[name] for name in
    ("unmatched_pickup_zone", "unmatched_dropoff_zone", "unmatched_weather_hour"))):
    raise ValueError(f"Gold quality failed: {checks.asDict()}")
print("Gold grain and relationships: PASS")